# Gold Layer: Analytics and Data Products

The Gold layer contains business-ready datasets built from the cleaned Silver layer.

For FlightPulse, the Gold layer will transform cleaned aircraft and flight data into datasets that can be used by dashboards, APIs, analytics, and future machine learning or AI features.

## Data Flow

```text
🥉 Bronze
     ↓
🥈 Silver
     ↓
🥇 Gold
     ↓
FastAPI → Angular website
```

The Gold layer is designed around the needs of the FlightPulse application rather than simply storing another copy of the Silver data.

## What it builds

| Tables | Source | Used by |
|---|---|---|
| `gold_flight_summary`, `gold_airline_performance`, `gold_airport_activity` | `silver_flights` (the latest night only) | The dashboard's Flights, Airlines and Airports tabs |
| `gold_insights_*` (6 tables) | `silver_flights_history` (the last 30 days) | The Insights page |

Live aircraft positions don't need a Gold table: the API reads `silver_aircraft` directly.

## Known limitations

- **Two definitions of "delayed":** the first three tables count any delay over 0 minutes, while
  the insight tables use the industry standard (over 15 minutes). The same airline can therefore
  look worse on the dashboard than on the Insights page.
- **The first three tables cover one night only**, not the history.
- **`gold_airport_activity` counts fetched flights, not the schedule:** only KUL and PEN
  departures are fetched, so other airports appear only as destinations, and the
  `scheduled_*` columns count the sampled landed flights.
- **Samples, not every flight:** all figures come from the nightly Aviationstack sample (see
  `01.1_ingest_AviationStack`), so small groups can swing a lot. The insight tables keep their
  flight counts for that reason.


### Step 1: Load Silver

In [ ]:
aircraft_df = spark.table("workspace.default.silver_aircraft")
flights_df = spark.table("workspace.default.silver_flights")

display(aircraft_df.limit(10))
display(flights_df.limit(10))

### Flight summary 

The `gold_flight_summary` table is an application-ready dataset containing the most important information about individual flights.

It is created from the cleaned `silver_flights` table and selects the fields most relevant to the FlightPulse website, API, dashboards, and future AI features.

Unlike the Silver layer, which focuses on cleaning and standardizing the source data, this Gold table is designed around how the application will consume the data.

#### Purpose

The table provides a simple view of each flight containing:

* Flight identification
* Airline information
* Departure and arrival airports
* Flight status
* Scheduled, estimated, and actual times
* Delay information
* Flight route

Each row represents **one flight**.

#### Source

```text
workspace.default.silver_flights
```

The Silver table is used as the source because it has already been cleaned, validated, and standardized.

#### Transformation

The Gold transformation selects the fields required by the application and creates additional derived fields.

The `route` field combines the departure and arrival IATA codes:

```text
HND → SIN
```

The delay indicators provide a simple way for downstream applications to determine whether a flight has a recorded delay.

* `true`: a delay greater than 0 minutes was recorded
* `false`: a delay of 0 minutes was recorded
* `null`: delay information is not available

This distinction is important because a missing delay value does not necessarily mean that a flight has no delay.

#### Main Fields

| Field                  | Description                                         |
| ---------------------- | --------------------------------------------------- |
| `flight_date`          | Date associated with the flight                     |
| `flight_status`        | Current status reported by Aviationstack            |
| `flight_iata`          | IATA flight identifier                              |
| `flight_icao`          | ICAO flight identifier                              |
| `flight_number`        | Flight number                                       |
| `airline_name`         | Airline operating the flight                        |
| `departure_airport`    | Departure airport name                              |
| `departure_iata`       | Departure airport IATA code                         |
| `arrival_airport`      | Arrival airport name                                |
| `arrival_iata`         | Arrival airport IATA code                           |
| `departure_delay`      | Recorded departure delay in minutes                 |
| `arrival_delay`        | Recorded arrival delay in minutes                   |
| `scheduled_departure`  | Scheduled departure time                            |
| `estimated_departure`  | Estimated departure time                            |
| `actual_departure`     | Actual departure time when available                |
| `scheduled_arrival`    | Scheduled arrival time                              |
| `estimated_arrival`    | Estimated arrival time                              |
| `actual_arrival`       | Actual arrival time when available                  |
| `route`                | Combined departure and arrival IATA codes           |
| `is_departure_delayed` | Indicates whether a recorded departure delay exists |
| `is_arrival_delayed`   | Indicates whether a recorded arrival delay exists   |

#### Why This Belongs in Gold

The Gold layer is intended for refined datasets that are tailored to analytics, dashboards, and applications. Databricks describes Gold as the layer where data is modeled around business or project requirements.


In [ ]:
from pyspark.sql.functions import *

gold_flight_summary = flights_df.select(
    "flight_date",
    "flight_status",
    "flight_iata",
    "flight_icao",
    "flight_number",
    "airline_name",
    "airline_iata",
    "airline_icao",
    "departure_airport",
    "departure_iata",
    "departure_icao",
    "arrival_airport",
    "arrival_iata",
    "arrival_icao",
    "departure_delay",
    "arrival_delay",
    "scheduled_departure",
    "estimated_departure",
    "actual_departure",
    "scheduled_arrival",
    "estimated_arrival",
    "actual_arrival"
)

display(gold_flight_summary.limit(10))

### Next step: add route and delay information

The Gold dataset is enhanced with:

* `route`: combines the departure and arrival IATA codes, for example `HND → SIN`.
* `is_departure_delayed`: indicates whether a recorded departure delay is greater than 0 minutes.
* `is_arrival_delayed`: indicates whether a recorded arrival delay is greater than 0 minutes.

A `null` delay remains `null` because missing delay information does not mean the flight was not delayed.


In [ ]:
gold_flight_summary = gold_flight_summary \
    .withColumn(
        "route",
        concat_ws(" → ", col("departure_iata"), col("arrival_iata"))
    ) \
    .withColumn(
        "is_departure_delayed",
        when(col("departure_delay").isNull(), None)
        .when(col("departure_delay") > 0, True)
        .otherwise(False)
    ) \
    .withColumn(
        "is_arrival_delayed",
        when(col("arrival_delay").isNull(), None)
        .when(col("arrival_delay") > 0, True)
        .otherwise(False)
    )

Now we can save the first Gold table: the flight summary.

In [ ]:
gold_flight_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_flight_summary")

In [ ]:
display(
    spark.table("workspace.default.gold_flight_summary").limit(10)
)

In [ ]:
from pyspark.sql.functions import *

gold_airline_performance = flights_df \
    .groupBy(
        "airline_name",
        "airline_iata",
        "airline_icao"
    ) \
    .agg(
        count("*").alias("total_flights"),
        sum(
            when(col("departure_delay") > 0, 1).otherwise(0)
        ).alias("delayed_departures"),
        avg("departure_delay").alias("average_departure_delay"),
        sum(
            when(col("arrival_delay") > 0, 1).otherwise(0)
        ).alias("delayed_arrivals"),
        avg("arrival_delay").alias("average_arrival_delay")
    )

display(gold_airline_performance)

In [ ]:
gold_airline_performance.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_airline_performance")

In [ ]:
display(
    spark.table("workspace.default.gold_airline_performance")
)

### Gold_airport_activity

This table summarizes flight activity by airport, including:

* Departures (`scheduled_departures`)
* Arrivals (`scheduled_arrivals`)
* Total activity

The counts are of the flights fetched from Aviationstack (landed departures from KUL and PEN),
not the airports' full schedules.

It will later support airport activity insights in FlightPulse.


In [ ]:
gold_airport_activity = flights_df \
    .select(
        col("departure_iata").alias("airport_iata"),
        col("departure_airport").alias("airport_name"),
        lit("departure").alias("activity_type")
    ) \
    .union(
        flights_df.select(
            col("arrival_iata").alias("airport_iata"),
            col("arrival_airport").alias("airport_name"),
            lit("arrival").alias("activity_type")
        )
    ) \
    .groupBy(
        "airport_iata",
        "airport_name"
    ) \
    .agg(
        count("*").alias("total_activity"),
        sum(
            when(col("activity_type") == "departure", 1).otherwise(0)
        ).alias("scheduled_departures"),
        sum(
            when(col("activity_type") == "arrival", 1).otherwise(0)
        ).alias("scheduled_arrivals")
    ) \
    .orderBy(desc("total_activity"))

display(gold_airport_activity)

In [ ]:
gold_airport_activity.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_airport_activity")

In [ ]:
display(
    spark.table("workspace.default.gold_airport_activity")
)

In [ ]:
display(spark.table("workspace.default.gold_flight_summary"))
display(spark.table("workspace.default.gold_airline_performance"))
display(spark.table("workspace.default.gold_airport_activity"))

### Delay insights

Turns `silver_flights_history` into small, ready-to-serve tables that answer
specific questions about departures from the tracked airports (KUL, PEN) over the
last 30 days.

### Definitions

| Term | Meaning |
|---|---|
| **Delay** | Actual minus scheduled departure time, in minutes (Aviationstack's own delay figure when the actual time is missing). Early departures count as 0. |
| **On time** | Departed no more than **15 minutes** after schedule, the standard industry definition. |
| **Delay band** | On time · 16–30 min · 31–60 min · Over 60 min |
| **Hour** | Scheduled departure hour in the airport's local time. |

Every table keeps its flight count, so small samples are visible; the website only
ranks groups with at least 5 flights.

| Table | Answers |
|---|---|
| `gold_insights_summary` | How punctual is each airport overall? Best and worst airline, worst hour |
| `gold_insights_by_airline` | Which airlines are most punctual? |
| `gold_insights_by_hour` | What time of day is worst to fly? |
| `gold_insights_by_route` | Which destinations see the most delays? |
| `gold_insights_daily` | How has punctuality changed day by day? |
| `gold_insights_delay_bands` | When flights are late, how late? |

In [ ]:
%sql
-- Aviationstack gives local airport times marked as +00:00, so read them in UTC
-- to keep the clock time (and therefore the local hour) unchanged.
SET TIME ZONE 'UTC';

-- One row per departure from a tracked airport in the last 30 days, with a delay we can trust.
CREATE OR REPLACE TEMP VIEW departures AS
WITH base AS (
  SELECT
    departure_iata AS airport,
    to_date(flight_date) AS flight_date,
    flight_iata,
    airline_name,
    arrival_iata,
    arrival_airport,
    hour(scheduled_departure) AS local_hour,
    greatest(
      coalesce(
        round((unix_timestamp(actual_departure) - unix_timestamp(scheduled_departure)) / 60),
        departure_delay
      ),
      0
    ) AS delay_min
  FROM workspace.default.silver_flights_history
  WHERE departure_iata IN ('KUL', 'PEN')
    AND to_date(flight_date) >= date_sub(current_date(), 30)
)
SELECT
  *,
  delay_min <= 15 AS on_time,
  CASE
    WHEN delay_min <= 15 THEN 'On time'
    WHEN delay_min <= 30 THEN '16-30 min'
    WHEN delay_min <= 60 THEN '31-60 min'
    ELSE 'Over 60 min'
  END AS delay_band
FROM base
WHERE delay_min IS NOT NULL;

CREATE OR REPLACE TABLE workspace.default.gold_insights_by_airline AS
SELECT
  airport,
  airline_name,
  count(*) AS flights,
  round(100 * avg(CAST(on_time AS INT)), 1) AS on_time_pct,
  round(avg(delay_min), 1) AS avg_delay_min,
  sum(CAST(delay_min > 60 AS INT)) AS over_60_min
FROM departures
WHERE airline_name IS NOT NULL
GROUP BY airport, airline_name;

CREATE OR REPLACE TABLE workspace.default.gold_insights_by_hour AS
SELECT
  airport,
  local_hour,
  count(*) AS flights,
  round(100 * avg(CAST(on_time AS INT)), 1) AS on_time_pct,
  round(avg(delay_min), 1) AS avg_delay_min
FROM departures
WHERE local_hour IS NOT NULL
GROUP BY airport, local_hour;

CREATE OR REPLACE TABLE workspace.default.gold_insights_by_route AS
SELECT
  airport,
  arrival_iata,
  first(arrival_airport, true) AS arrival_airport,
  count(*) AS flights,
  round(100 * avg(CAST(on_time AS INT)), 1) AS on_time_pct,
  round(avg(delay_min), 1) AS avg_delay_min
FROM departures
WHERE arrival_iata IS NOT NULL
GROUP BY airport, arrival_iata;

CREATE OR REPLACE TABLE workspace.default.gold_insights_daily AS
SELECT
  airport,
  flight_date,
  count(*) AS flights,
  round(100 * avg(CAST(on_time AS INT)), 1) AS on_time_pct,
  round(avg(delay_min), 1) AS avg_delay_min
FROM departures
GROUP BY airport, flight_date;

CREATE OR REPLACE TABLE workspace.default.gold_insights_delay_bands AS
SELECT
  airport,
  delay_band,
  count(*) AS flights,
  round(100 * count(*) / sum(count(*)) OVER (PARTITION BY airport), 1) AS share_pct
FROM departures
GROUP BY airport, delay_band;

-- One headline row per airport. Best/worst airline and worst hour need at least 5 flights.
CREATE OR REPLACE TABLE workspace.default.gold_insights_summary AS
WITH overall AS (
  SELECT
    airport,
    count(*) AS flights,
    count(DISTINCT flight_date) AS days,
    min(flight_date) AS first_day,
    max(flight_date) AS last_day,
    round(100 * avg(CAST(on_time AS INT)), 1) AS on_time_pct,
    round(avg(delay_min), 1) AS avg_delay_min
  FROM departures
  GROUP BY airport
),
airlines AS (
  SELECT
    airport,
    airline_name,
    on_time_pct,
    row_number() OVER (PARTITION BY airport ORDER BY on_time_pct DESC, flights DESC) AS best_rank,
    row_number() OVER (PARTITION BY airport ORDER BY on_time_pct ASC, flights DESC) AS worst_rank
  FROM workspace.default.gold_insights_by_airline
  WHERE flights >= 5
),
hours AS (
  SELECT
    airport,
    local_hour,
    row_number() OVER (PARTITION BY airport ORDER BY on_time_pct ASC, flights DESC) AS worst_rank
  FROM workspace.default.gold_insights_by_hour
  WHERE flights >= 5
)
SELECT
  o.*,
  best.airline_name AS most_punctual_airline,
  best.on_time_pct AS most_punctual_on_time_pct,
  worst.airline_name AS least_punctual_airline,
  worst.on_time_pct AS least_punctual_on_time_pct,
  h.local_hour AS worst_hour
FROM overall o
LEFT JOIN airlines best ON best.airport = o.airport AND best.best_rank = 1
LEFT JOIN airlines worst ON worst.airport = o.airport AND worst.worst_rank = 1
LEFT JOIN hours h ON h.airport = o.airport AND h.worst_rank = 1;

SELECT * FROM workspace.default.gold_insights_summary;
